In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/processed/amsterdam_listings_cleaned.csv')
print(df.shape)
df.head()

(6356, 70)


,id,name,description,host_id,host_name,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_is_superhost,...,has_air_conditioning,has_free_parking,has_pool,has_dedicated_workspace,has_tv,has_reviews,host_info_missing,has_host_about,has_host_location,has_license
0,28871,Comfortable double room,Basic bedroom in the center of Amsterdam.,124245,Edwin,16.0,1.0,15.0,5.0,t,...,False,False,False,False,False,True,False,True,True,True
1,44129,Luxury design with canal view,"Welcome to my little gem<br /><br />Cozy, brig...",187728,Tanya,15.0,10.0,15.0,5.0,t,...,False,False,False,False,False,True,False,True,True,True
2,49552,Multatuli Luxury Guest Suite in top location,Stylish & spacious 60m2 guest suite in Amsterd...,225987,Joanna & MP,15.0,9.0,15.0,5.0,t,...,False,False,False,True,False,True,False,True,True,True
3,50263,Central de Lux 2 bedrooms (4p) apt 125 sqm,A beautiful 'De Lux' 125 sqm apartment for 4 a...,230246,Donald,15.0,9.0,15.0,5.0,f,...,False,False,False,True,False,True,False,True,True,True
4,50523,B & B de 9 Straatjes (city center),B&B “De 9 Straatjes” – Your home in the heart ...,231946,Raymond,15.0,9.0,15.0,5.0,t,...,False,False,False,False,False,True,False,True,True,True


**Calculate distance to Amsterdam's center**

We'll use the Haversine formula — a standard way to calculate the distance between two points on Earth given their latitude/longitude (accounts for the Earth's curvature, unlike a simple straight-line calculation).

In [8]:
import numpy as np


# What this does: for every listing, it calculates the straight-line distance (in km) from that listing's coordinates to Dam Square, using the Haversine formula (a well-known geospatial calculation, not something we invented). The result is a single number per listing — smaller = more central, larger = further out.


def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

center_lat, center_lon = 52.3676, 4.9041

df['distance_to_center'] = haversine_distance(df['latitude'], df['longitude'], center_lat, center_lon)

print(df['distance_to_center'])

0       0.895008
1       2.015749
2       1.670972
3       1.741133
4       1.367068
          ...   
6351    1.905714
6352    1.447927
6353    1.542824
6354    1.457429
6355    0.741142
Name: distance_to_center, Length: 6356, dtype: float64



count    6356.000000
mean        2.760206
std         1.610042
min         0.120076
25%         1.661794
50%         2.458791
75%         3.476530
max        10.217935
Name: distance_to_center, dtype: float64

In [9]:
df['distance_to_center'].describe()

count    6356.000000
mean        2.760206
std         1.610042
min         0.120076
25%         1.661794
50%         2.458791
75%         3.476530
max        10.217935
Name: distance_to_center, dtype: float64

**Host-listing-count buckets — your narrative feature**

Now let's build the feature tied to your project's actual differentiator: distinguishing genuine home-sharers from professional/business hosts.


In [12]:
df['host_listings_count'].describe()


count    6356.000000
mean        5.516205
std        53.283295
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max      1167.000000
Name: host_listings_count, dtype: float64

What we're doing: building the feature that makes your project's story (the "genuine home-sharer vs. professional/business host" angle) actually real, not just a sentence in your README.

The column host_listings_count tells you how many total listings a host has across all of Airbnb. A host with 1 is renting out a single spare room/apartment — that's the "genuine home-sharer" Airbnb was originally meant for. A host with 50 or 1167 listings is clearly running a business — probably a property management company, not a person renting their spare room.

In [13]:
# See how hosts distribute across candidate bucket ranges

bins = [0, 1, 2, 5, 10, 20, 50, 1200]
labels = ['1', '2', '3-5', '6-10', '11-20', '21-50', '50+']

host_bucket_check = pd.cut(df['host_listings_count'], bins=bins, labels=labels)
print(host_bucket_check.value_counts().sort_index())

# This slices host_listings_count into candidate ranges and counts how many listings fall into each — just so we can see the distribution before finalizing our actual buckets.

host_listings_count
1        4627
2         698
3-5       504
6-10      242
11-20     142
21-50     112
50+        31
Name: count, dtype: int64


This gives us a clean, defensible way to group things. Here's the read:

 * 1 listing: 4,627 (73%) — the clear majority, genuine individual hosts
 * 2-5 listings: 1,202 (19%) — small-scale hosts, maybe a couple properties
 * 6-20 listings: 384 (6%) — this is where it starts looking like a small business, not a hobby
 * 21+ listings: 143 (2%) — unmistakably commercial/professional operations

In [14]:
# Create the actual bucket column

def host_bucket(count):
    if count == 1:
        return 'Single host'
    elif count <= 5:
        return 'Small multi-host'
    elif count <= 20:
        return 'Professional host'
    else:
        return 'Large/commercial host'

df['host_type'] = df['host_listings_count'].apply(host_bucket)

df['host_type'].value_counts()

host_type
Single host              4627
Small multi-host         1202
Professional host         384
Large/commercial host     143
Name: count, dtype: int64

In [15]:
df.groupby('host_type')['price'].median().sort_values()

host_type
Large/commercial host    199.71
Professional host        219.00
Small multi-host         239.00
Single host              312.00
Name: price, dtype: float64

In [16]:
# Encode room_type and property_type

print(df['room_type'].value_counts())
print()
print(df['property_type'].value_counts())

room_type
Entire home/apt    4765
Private room       1551
Shared room          21
Hotel room           19
Name: count, dtype: int64

property_type
Entire rental unit                    2455
Entire condo                           905
Entire home                            706
Room in hotel                          331
Private room in rental unit            287
Private room in bed and breakfast      259
Private room in condo                  151
Entire loft                            141
Houseboat                              126
Entire townhouse                       111
Private room in home                   109
Entire serviced apartment               91
Boat                                    90
Room in boutique hotel                  77
Private room in townhouse               65
Private room in houseboat               62
Private room in guest suite             55
Entire guesthouse                       37
Entire guest suite                      36
Private room in boat                

* room_type (already clean) → one-hot encode directly, no cleanup needed
* property_type → too fragmented to use as-is. A model can't learn anything meaningful from a category with 1 row. We'll simplify it into broader groups first, then encode.

In [17]:
# Simplify property_type into broader groups


def simplify_property_type(prop_type):
    prop_type = prop_type.lower()
    if 'houseboat' in prop_type or 'boat' in prop_type:
        return 'Houseboat/Boat'
    elif 'hotel' in prop_type or 'bed and breakfast' in prop_type or 'aparthotel' in prop_type:
        return 'Hotel/B&B'
    elif 'condo' in prop_type or 'rental unit' in prop_type or 'serviced apartment' in prop_type or 'loft' in prop_type or 'guest suite' in prop_type:
        return 'Apartment'
    elif 'home' in prop_type or 'house' in prop_type or 'townhouse' in prop_type or 'villa' in prop_type or 'cottage' in prop_type or 'bungalow' in prop_type or 'cabin' in prop_type:
        return 'House'
    elif 'hostel' in prop_type:
        return 'Hostel'
    else:
        return 'Other'

df['property_type_grouped'] = df['property_type'].apply(simplify_property_type)

df['property_type_grouped'].value_counts()

property_type_grouped
Apartment         4167
House             1114
Hotel/B&B          701
Houseboat/Boat     314
Other               49
Hostel              11
Name: count, dtype: int64

**One-hot encode room_type and property_type_grouped**

In [18]:
df = pd.get_dummies(df, columns=['room_type', 'property_type_grouped'], drop_first=True)
df.shape

(6356, 79)

In [20]:
# Encode neighbourhood_cleansed

df['neighbourhood_cleansed'].value_counts()

neighbourhood_cleansed
De Baarsjes - Oud-West                    1064
Centrum-West                               844
De Pijp - Rivierenbuurt                    695
Centrum-Oost                               664
Zuid                                       432
Westerpark                                 402
Oud-Oost                                   383
Oud-Noord                                  278
Bos en Lommer                              257
Oostelijk Havengebied - Indische Buurt     231
Noord-West                                 205
Watergraafsmeer                            178
Noord-Oost                                 123
Slotervaart                                117
IJburg - Zeeburgereiland                   116
Geuzenveld - Slotermeer                     94
Buitenveldert - Zuidas                      87
De Aker - Nieuw Sloten                      46
Bijlmer-Centrum                             43
Osdorp                                      40
Gaasperdam - Driemond                

In [21]:
# One-hot encode neighbourhood_cleansed

df = pd.get_dummies(df, columns=['neighbourhood_cleansed'], drop_first=True)
df.shape

(6356, 99)

In [26]:
# Fix boolean columns

df.dtypes.value_counts()


bool       42
float64    29
int64      18
str        10
Name: count, dtype: int64

In [30]:
# some fixes

df['bathrooms_text'].unique()

<StringArray>
[    '1 shared bath',         '1.5 baths',            '1 bath',
    '1 private bath',           '2 baths',         '2.5 baths',
  '1.5 shared baths',           '3 baths',         '3.5 baths',
           '4 baths',    '0 shared baths',           '0 baths',
    '2 shared baths',         'Half-bath', 'Private half-bath',
         '5.5 baths',    '3 shared baths',          '12 baths',
         '4.5 baths',           '5 baths',          '17 baths',
  'Shared half-bath']
Length: 22, dtype: str

In [31]:
import re

def extract_bathroom_count(text):
    if 'half-bath' in text.lower():
        return 0.5
    match = re.search(r'(\d+\.?\d*)', text)
    return float(match.group(1)) if match else np.nan

df['bathrooms_num'] = df['bathrooms_text'].apply(extract_bathroom_count)
df['bathrooms_shared'] = df['bathrooms_text'].str.contains('shared', case=False)

print(df[['bathrooms_text', 'bathrooms_num', 'bathrooms_shared']].drop_duplicates())


# Handles the two "half-bath" cases specially (no number in the text, so we manually assign 0.5)
# For everything else, uses a regex (pattern-matching for numbers) to pull out the numeric value, e.g., "1.5 baths" → 1.5
# Separately checks if the word "shared" appears, creating a True/False column for that

         bathrooms_text  bathrooms_num  bathrooms_shared
0         1 shared bath            1.0              True
1             1.5 baths            1.5             False
2                1 bath            1.0             False
6        1 private bath            1.0             False
8               2 baths            2.0             False
23            2.5 baths            2.5             False
49     1.5 shared baths            1.5              True
60              3 baths            3.0             False
143           3.5 baths            3.5             False
236             4 baths            4.0             False
257      0 shared baths            0.0              True
268             0 baths            0.0             False
347      2 shared baths            2.0              True
459           Half-bath            0.5             False
541   Private half-bath            0.5             False
1387          5.5 baths            5.5             False
1420     3 shared baths        

**Final feature selection — decide what goes into the model**

This is the last step of feature engineering: separating columns into three groups:

* Actual model features (numeric/boolean columns that should train the model)
* Reference-only columns (things like id, name, description, host_name — useful to keep in the dataframe for context/debugging, but shouldn't be fed into the model)
* The target (price — needs to be separate from the features)

In [27]:
list(df.columns)

['id',
 'name',
 'description',
 'host_id',
 'host_name',
 'hosts_time_as_user_years',
 'hosts_time_as_user_months',
 'hosts_time_as_host_years',
 'hosts_time_as_host_months',
 'host_is_superhost',
 'host_listings_count',
 'host_has_profile_pic',
 'host_identity_verified',
 'latitude',
 'longitude',
 'property_type',
 'accommodates',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'has_availability',
 'availability_30',
 'availability_60',
 'availability_90',
 'availability_365',
 'number_of_reviews',
 'number_of_reviews_ltm',
 'number_of_reviews_l30d',
 'availability_eoy',
 'number_of_reviews_ly',
 'estimated_occupancy_l365d',
 'estimated_revenue_l365d',
 'review_scores_rating',
 'review_scores_accuracy',
 'review_scores_cleanliness',
 'review_scores_checkin',
 'review_scores_communic

In [32]:
# Drop the original text column, encode host_type

df = df.drop(columns=['bathrooms_text', 'property_type'])

df = pd.get_dummies(df, columns=['host_type'], drop_first=True)

df.shape

(6356, 101)

In [33]:
# Split into features (X), target (y), and reference columns

# Target - what we're predicting
y = df['price']

# Reference columns - keep for context/debugging, never feed to the model
reference_cols = ['id', 'name', 'description', 'host_id', 'host_name']

# Columns to exclude from features: target, reference, and the leakage risk
exclude_cols = ['price', 'price_per_person'] + reference_cols

# Everything else becomes X
X = df.drop(columns=exclude_cols)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nAny non-numeric columns left in X?")
print(X.dtypes[~X.dtypes.isin(['int64', 'float64', 'bool'])])

X shape: (6356, 94)
y shape: (6356,)

Any non-numeric columns left in X?
hosts_time_as_user_years                        float64
hosts_time_as_user_months                       float64
hosts_time_as_host_years                        float64
hosts_time_as_host_months                       float64
host_is_superhost                                   str
host_listings_count                             float64
host_has_profile_pic                                str
host_identity_verified                              str
latitude                                        float64
longitude                                       float64
accommodates                                      int64
bedrooms                                        float64
beds                                            float64
minimum_nights                                  float64
maximum_nights                                  float64
minimum_minimum_nights                          float64
maximum_minimum_nights         

In [34]:
# some fixes

bool_text_cols = ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'has_availability']

for col in bool_text_cols:
    X[col] = X[col].map({'t': True, 'f': False})

# Correct check this time - using astype(str) for reliable comparison
print(X.dtypes.astype(str).value_counts())

bool       47
float64    28
int64      16
object      3
Name: count, dtype: int64


In [35]:
for col in bool_text_cols:
    print(col, X[col].unique())


host_is_superhost [True False nan]
host_has_profile_pic [True False nan]
host_identity_verified [True False nan]
has_availability [ True False]


In [36]:
# Fix the leftover NaNs and lock in proper bool type

for col in ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified']:
    X[col] = X[col].fillna(False).astype(bool)

print(X.dtypes.astype(str).value_counts())

bool       50
float64    28
int64      16
Name: count, dtype: int64


Save the final feature-engineered dataset

In [37]:
X.to_csv('../data/processed/X_features.csv', index=False)
y.to_csv('../data/processed/y_target.csv', index=False)

print(f"Saved X: {X.shape}, y: {y.shape}")

Saved X: (6356, 94), y: (6356,)
